# GTE Embeddings & Vector Search on Databricks

This notebook demonstrates an end-to-end workflow for:

1. **Querying the `databricks-gte-large-en` Foundation Model API** to generate 1024-dimension text embeddings using three different methods (MLflow SDK, Databricks SDK, and SQL)
2. **Comparing embeddings** using cosine similarity to verify semantic relevance
3. **Creating a Vector Search index** backed by GTE embeddings for real-time semantic search
4. **Querying the Vector Search index** with natural language questions

## Prerequisites
- Databricks workspace with Foundation Model APIs enabled
- The `databricks-gte-large-en` pay-per-token endpoint (pre-configured, no setup needed)
- Permission to create tables in a Unity Catalog schema

---

# Part I: Querying GTE Embeddings

Before building a Vector Search index, let's first understand how to generate embeddings using the `databricks-gte-large-en` Foundation Model API. We'll explore three different methods and verify the embeddings capture semantic meaning.

## Step 1: Query GTE Embeddings with MLflow Deployments SDK

The simplest way to query the `databricks-gte-large-en` endpoint is through the **MLflow Deployments SDK**. This endpoint is a pre-configured pay-per-token Foundation Model API — no endpoint creation needed.

The GTE Large model maps any text to a **1024-dimension embedding vector** with an 8192-token context window, suitable for retrieval, classification, clustering, and semantic search.

In [0]:
import mlflow.deployments

# Create a deployment client for Databricks
client = mlflow.deployments.get_deploy_client("databricks")

# Query the GTE Large endpoint with a single text
response = client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": "Databricks is a unified analytics platform for data engineering and machine learning."},
)

# Extract the embedding vector
embedding = response["data"][0]["embedding"]
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1024
First 5 values: [0.1256103515625, -0.28564453125, -0.1959228515625, 0.12939453125, 0.1494140625]


## Step 2: Query GTE Embeddings with Databricks SDK

The **Databricks SDK for Python** provides an alternative way to query the same endpoint. The key difference from the MLflow SDK is the response format — the SDK returns a `QueryEndpointResponse` object (access via `.data[0].embedding`) instead of a dictionary (access via `["data"][0]["embedding"]`).

For embeddings models, the `input` parameter must be a **string or list of strings**, not a dictionary.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Query the GTE Large endpoint using the Databricks SDK
# For embeddings models, 'input' must be a string or list of strings
response = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input=["Machine learning models can be served at scale on Databricks."],
)

embedding = response.data[0].embedding
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimension: 1024
First 5 values: [0.1143798828125, -0.42431640625, -0.79638671875, 0.29541015625, 0.08038330078125]


## Step 3: Query GTE Embeddings with SQL `ai_query`

You can also generate embeddings **directly in SQL** using the built-in `ai_query()` function. This is useful for embedding text in data pipelines, batch processing, or when working with SQL warehouses.

The function takes two arguments:
- The endpoint name (`'databricks-gte-large-en'`)
- The text to embed

It returns an `ARRAY<DOUBLE>` containing the 1024-dimension embedding vector.

In [0]:
%sql
-- Use the ai_query SQL function to generate embeddings directly in SQL
-- ai_query() returns an ARRAY<DOUBLE> containing the 1024-dimension embedding vector
SELECT ai_query(
  'databricks-gte-large-en',
  'Databricks Unity Catalog provides unified governance for data and AI assets.'
) AS embedding;

embedding


## Step 4: Compare Embeddings with Cosine Similarity

Now let's verify that the embeddings make **semantic sense**. We'll embed three texts — two about technology and one about food — and compute cosine similarity between each pair.

**Expected result:** Tech texts should have higher similarity with each other than with the food text, because GTE captures semantic meaning, not just keyword overlap.

In [0]:
import numpy as np

# Multiple texts to embed and compare
texts = [
    "Databricks is a unified analytics platform.",
    "Apache Spark is a distributed computing engine.",
    "I love eating pizza on weekends.",
]

response = client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": texts},
)

embeddings = [item["embedding"] for item in response["data"]]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Each embedding dimension: {len(embeddings[0])}")

# Compute cosine similarity between pairs
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nCosine Similarity:")
print(f"  Text 0 vs Text 1 (both about tech): {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  Text 0 vs Text 2 (tech vs food):    {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"  Text 1 vs Text 2 (tech vs food):    {cosine_similarity(embeddings[1], embeddings[2]):.4f}")

Number of embeddings: 3
Each embedding dimension: 1024

Cosine Similarity:
  Text 0 vs Text 1 (both about tech): 0.8320
  Text 0 vs Text 2 (tech vs food):    0.5366
  Text 1 vs Text 2 (tech vs food):    0.5060


---

# Part II: Vector Search with GTE Embeddings

Now we'll use the GTE embeddings to power a **Vector Search index** for real-time semantic search. This involves:

1. Creating a source table with documents to index
2. Creating a Vector Search endpoint (the compute that powers similarity search)
3. Creating a Delta Sync Index that auto-embeds the text column using `databricks-gte-large-en`
4. Querying the index with natural language questions

## Step 5: Create a Source Table

We'll create a table with 10 sample documents — 7 about Databricks technologies and 3 about food. This mix will let us verify that semantic search correctly distinguishes between topics.

In [0]:
%sql
-- Create a source table with sample documents for Vector Search
CREATE OR REPLACE TABLE dbacademy.default.documents_for_vector_search (
  id BIGINT,
  title STRING,
  text STRING
);

INSERT INTO dbacademy.default.documents_for_vector_search VALUES
  (1, 'Databricks Platform', 'Databricks is a unified analytics platform for data engineering, data science, and machine learning. It provides a collaborative environment for processing and analyzing large datasets using Apache Spark.'),
  (2, 'Apache Spark', 'Apache Spark is a distributed computing engine for large-scale data processing. It supports batch processing, streaming, machine learning, and graph computation.'),
  (3, 'Delta Lake', 'Delta Lake is an open-source storage layer that brings ACID transactions to Apache Spark and big data workloads. It provides reliability, performance, and lifecycle management for data lakes.'),
  (4, 'Unity Catalog', 'Unity Catalog is a unified governance solution for data and AI assets on Databricks. It provides centralized access control, auditing, lineage, and data discovery across all workspaces.'),
  (5, 'MLflow', 'MLflow is an open-source platform for managing the end-to-end machine learning lifecycle. It supports experiment tracking, model packaging, model registry, and deployment.'),
  (6, 'Model Serving', 'Databricks Model Serving provides real-time inference for machine learning models. It supports autoscaling, scale-to-zero, and GPU workloads for production deployment.'),
  (7, 'Vector Search', 'Databricks Vector Search is a similarity search engine that enables fast and accurate retrieval of relevant data using vector embeddings. It integrates with Foundation Model APIs for automatic embedding generation.'),
  (8, 'Italian Food', 'Pizza is a popular Italian dish made with a flatbread base topped with tomato sauce, cheese, and various ingredients. It is enjoyed worldwide in many variations.'),
  (9, 'French Cuisine', 'Croissants are a buttery, flaky pastry of Austrian origin but most associated with France. They are a staple of French breakfast and bakery culture.'),
  (10, 'Japanese Cooking', 'Sushi is a traditional Japanese dish of prepared vinegared rice accompanied by seafood, vegetables, and occasionally tropical fruits. It is an art form in Japanese cuisine.');

SELECT * FROM dbacademy.default.documents_for_vector_search;

## Step 6: Create a Vector Search Endpoint

A **Vector Search endpoint** is the compute resource that powers similarity search queries. It hosts the index and handles ANN (Approximate Nearest Neighbor) searches.

We use the `STANDARD` endpoint type, which is suitable for most workloads. Endpoint creation takes a few minutes to provision.

In [0]:
from databricks.sdk.service.vectorsearch import EndpointType

endpoint_name = "vector_search_demo_endpoint"

# Create the Vector Search endpoint
print(f"Creating Vector Search endpoint: {endpoint_name}...")
w.vector_search_endpoints.create_endpoint(
    name=endpoint_name,
    endpoint_type=EndpointType.STANDARD,
)
print("Endpoint creation initiated.")

# Wait for the endpoint to come online
import time
print("Waiting for endpoint to be online...")
for i in range(60):
    endpoints = list(w.vector_search_endpoints.list_endpoints())
    for ep in endpoints:
        if ep.name == endpoint_name:
            state = ep.endpoint_status.state
            print(f"  Attempt {i+1}: state={state}")
            if state.name == "ONLINE":
                print("✅ Endpoint is online!")
                break
    else:
        time.sleep(10)
        continue
    break

## Step 7: Create the Vector Search Index

We create a **Delta Sync Index** that automatically:
1. Reads the `text` column from our source table
2. Sends it to the `databricks-gte-large-en` endpoint for embedding
3. Stores the 1024-dimension vectors in the index
4. Enables similarity search on the embedded content

We use `TRIGGERED` sync mode (manual sync) instead of `CONTINUOUS` (auto-sync on every change). For production, use `CONTINUOUS` so the index updates automatically when the source table changes.

In [0]:
from databricks.sdk.service.vectorsearch import (
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
)

index_name = "dbacademy.default.documents_vector_index"

# Create a Delta Sync Index with GTE embeddings
print(f"Creating Vector Search index: {index_name}...")
index = w.vector_search_indexes.create_index(
    name=index_name,
    endpoint_name=endpoint_name,
    primary_key="id",
    index_type=VectorIndexType.DELTA_SYNC,
    delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
        source_table="dbacademy.default.documents_for_vector_search",
        pipeline_type=PipelineType.TRIGGERED,
        embedding_source_columns=[
            EmbeddingSourceColumn(
                name="text",
                embedding_model_endpoint_name="databricks-gte-large-en",
            )
        ],
    ),
)

print(f"Index created! Ready: {index.status.ready}")
print(f"Message: {index.status.message}")

## Step 8: Wait for Index to Be Ready

The index needs time to provision pipeline resources and sync the initial data. The GTE endpoint embeds each document, and the index stores the vectors for fast similarity search.

This typically takes 5–10 minutes for the first index on a new endpoint.

In [0]:
# Wait for the index to be ready and sync initial data
print("Waiting for index to be ready...")
for i in range(60):
    idx = w.vector_search_indexes.get_index(index_name=index_name)
    ready = idx.status.ready
    rows = idx.status.indexed_row_count
    msg = idx.status.message
    print(f"  Attempt {i+1}: ready={ready}, rows={rows}, msg={msg}")
    if ready:
        print("✅ Index is ready with", rows, "documents indexed!")
        break
    time.sleep(15)
else:
    print("⚠️ Timed out waiting. Check the index status in the UI.")

## Step 9: Query the Vector Search Index (Semantic Search)

Now the fun part — we can search the index using **natural language questions**. The Vector Search endpoint:
1. Embeds the query text using the same GTE model
2. Finds the most similar documents by comparing embedding vectors
3. Returns ranked results with similarity scores

We'll run three queries that test different semantic domains:
- A tech question → should return tech documents
- A food question → should return food documents
- A deployment question → should return Model Serving and MLflow docs

In [0]:
# Query the Vector Search index with natural language questions
queries = [
    "Tell me about data analytics platforms",
    "What is a popular Italian dish?",
    "How to deploy machine learning models?",
]

print("=== Semantic Search Results ===\n")
for query in queries:
    print(f"Query: '{query}'")
    print("-" * 60)
    
    result = w.vector_search_indexes.query_index(
        index_name=index_name,
        columns=["id", "title", "text"],
        query_text=query,
        num_results=3,
    )
    
    # Extract column names from the manifest
    columns = [c.name for c in result.manifest.columns]
    for row in result.result.data_array:
        doc_id = int(row[columns.index("id")])
        title = row[columns.index("title")]
        score = row[columns.index("score")]
        print(f"  Score: {score:.4f} | ID: {doc_id} | {title}")
    print()

---

# Summary

## What We Built

| Component | Name | Purpose |
|-----------|------|---------|
| Source Table | `dbacademy.default.documents_for_vector_search` | 10 sample documents (tech + food) |
| Vector Search Endpoint | `vector_search_demo_endpoint` | Compute for similarity search |
| Vector Search Index | `dbacademy.default.documents_vector_index` | Delta Sync Index with GTE embeddings |
| Embedding Model | `databricks-gte-large-en` | Pay-per-token Foundation Model API |

## Key Takeaways

1. The `databricks-gte-large-en` endpoint is **pre-configured** — no setup needed, just query it
2. GTE produces **1024-dimension embeddings** that capture semantic meaning
3. Cosine similarity confirms tech texts score higher with each other (~0.83) than with food texts (~0.52)
4. Vector Search auto-embeds the `text` column using the GTE endpoint — no manual embedding needed
5. Semantic search correctly returns relevant documents regardless of keyword overlap

## To Add More Documents

```python
# Insert new rows into the source table, then trigger a sync
spark.sql("INSERT INTO dbacademy.default.documents_for_vector_search VALUES (11, 'New Doc', 'text here')")
w.vector_search_indexes.sync_index(index_name=index_name)
```

## Cleanup (Optional)

```python
# Delete the index
w.vector_search_indexes.delete_index(index_name=index_name)
# Delete the endpoint
w.vector_search_endpoints.delete_endpoint(name=endpoint_name)
# Drop the source table
spark.sql("DROP TABLE IF EXISTS dbacademy.default.documents_for_vector_search")
```